In [ ]:
# 통합 예제: 코퍼스 → 전처리 → 문서 표현 → TF-IDF → 정규화 → 코사인 유사도

import re
import math
import numpy as np
from collections import Counter

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity


# 1. 코퍼스 준비
corpus = [
    "AI technology is changing the world.",
    "Machine learning is a core technology of AI.",
    "Soccer and baseball are popular sports.",
    "Deep learning models are used in AI systems."
]

print("===== 1. 원본 코퍼스 =====")
for i, doc in enumerate(corpus):
    print(f"문서 {i+1}: {doc}")


# 2. 정규화 함수
def clean_text(text):
    text = text.lower()                  # 소문자 변환
    text = re.sub(r"[^a-z\s]", "", text) # 영문자와 공백만 남김
    text = re.sub(r"\s+", " ", text)     # 여러 공백 정리
    return text.strip()


cleaned_corpus = [clean_text(doc) for doc in corpus]

print("\n===== 2. 정규화 후 문서 =====")
for i, doc in enumerate(cleaned_corpus):
    print(f"문서 {i+1}: {doc}")


# 3. 토큰화
tokenized_corpus = [doc.split() for doc in cleaned_corpus]

print("\n===== 3. 토큰화 결과 =====")
for i, tokens in enumerate(tokenized_corpus):
    print(f"문서 {i+1}: {tokens}")


# 4. 불용어 제거
stopwords = {"is", "a", "the", "of", "and", "are", "in"}

filtered_corpus = [
    [token for token in tokens if token not in stopwords]
    for tokens in tokenized_corpus
]

print("\n===== 4. 불용어 제거 결과 =====")
for i, tokens in enumerate(filtered_corpus):
    print(f"문서 {i+1}: {tokens}")


# 5. 코퍼스 기본 통계
all_tokens = [token for doc in filtered_corpus for token in doc]
T = len(all_tokens)              # 전체 토큰 수
V = set(all_tokens)              # 고유 단어 집합
V_size = len(V)                  # 고유 단어 수
TTR = V_size / T                 # 타입/토큰 비율

print("\n===== 5. 코퍼스 통계 =====")
print("전체 토큰 수 T:", T)
print("고유 단어 수 V:", V_size)
print("TTR:", round(TTR, 3))
print("단어 빈도:", Counter(all_tokens))


# 6. 직접 BOW 벡터 만들기
vocab = sorted(list(V))

print("\n===== 6. 어휘 사전 =====")
print(vocab)

bow_matrix = []

for doc in filtered_corpus:
    bow_vector = [doc.count(word) for word in vocab]
    bow_matrix.append(bow_vector)

bow_matrix = np.array(bow_matrix)

print("\n===== 7. 직접 만든 BOW 행렬 =====")
print(bow_matrix)


# 7. TF, DF, IDF 직접 계산
def compute_tf(term, document):
    return document.count(term)


def compute_df(term, docs):
    return sum(1 for doc in docs if term in doc)


def compute_idf(term, docs):
    N = len(docs)
    df = compute_df(term, docs)
    return math.log((N + 1) / (df + 1)) + 1


print("\n===== 8. TF / DF / IDF 직접 계산 =====")
for word in vocab:
    tf_doc1 = compute_tf(word, filtered_corpus[0])
    df = compute_df(word, filtered_corpus)
    idf = compute_idf(word, filtered_corpus)

    print(f"{word:12s} TF(문서1): {tf_doc1}, DF: {df}, IDF: {idf:.3f}")


# 8. sklearn CountVectorizer
joined_corpus = [" ".join(doc) for doc in filtered_corpus]

count_vec = CountVectorizer()
count_matrix = count_vec.fit_transform(joined_corpus)

print("\n===== 9. CountVectorizer 결과 =====")
print("단어 목록:", count_vec.get_feature_names_out())
print(count_matrix.toarray())


# 9. sklearn TfidfVectorizer
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(joined_corpus)

print("\n===== 10. TF-IDF 결과 =====")
print("단어 목록:", tfidf_vec.get_feature_names_out())
print(np.round(tfidf_matrix.toarray(), 3))


# 10. 벡터 정규화
vectors = np.array([
    [3, 4],
    [1, 2],
    [10, 0]
])

l1_normalized = normalize(vectors, norm="l1")
l2_normalized = normalize(vectors, norm="l2")
max_normalized = vectors / vectors.max(axis=1, keepdims=True)

print("\n===== 11. 벡터 정규화 =====")
print("원본 벡터:")
print(vectors)

print("\nL1 정규화:")
print(np.round(l1_normalized, 3))

print("\nL2 정규화:")
print(np.round(l2_normalized, 3))

print("\nMax 정규화:")
print(np.round(max_normalized, 3))


# 11. 코사인 유사도 계산
sim_matrix = cosine_similarity(tfidf_matrix)

print("\n===== 12. 문서 간 코사인 유사도 =====")
print(np.round(sim_matrix, 3))


# 12. 문서 유사도 검색
query = "AI learning technology"
query_cleaned = clean_text(query)

query_tfidf = tfidf_vec.transform([query_cleaned])
similarities = cosine_similarity(query_tfidf, tfidf_matrix).reshape(-1)

top_index = np.argsort(-similarities)

print("\n===== 13. 문서 유사도 검색 =====")
print("검색 문장:", query)

for idx in top_index:
    print(f"문서 {idx+1} / 유사도 {similarities[idx]:.3f}")
    print("내용:", corpus[idx])
    print()

===== 1. 원본 코퍼스 =====
문서 1: AI technology is changing the world.
문서 2: Machine learning is a core technology of AI.
문서 3: Soccer and baseball are popular sports.
문서 4: Deep learning models are used in AI systems.

===== 2. 정규화 후 문서 =====
문서 1: ai technology is changing the world
문서 2: machine learning is a core technology of ai
문서 3: soccer and baseball are popular sports
문서 4: deep learning models are used in ai systems

===== 3. 토큰화 결과 =====
문서 1: ['ai', 'technology', 'is', 'changing', 'the', 'world']
문서 2: ['machine', 'learning', 'is', 'a', 'core', 'technology', 'of', 'ai']
문서 3: ['soccer', 'and', 'baseball', 'are', 'popular', 'sports']
문서 4: ['deep', 'learning', 'models', 'are', 'used', 'in', 'ai', 'systems']

===== 4. 불용어 제거 결과 =====
문서 1: ['ai', 'technology', 'changing', 'world']
문서 2: ['machine', 'learning', 'core', 'technology', 'ai']
문서 3: ['soccer', 'baseball', 'popular', 'sports']
문서 4: ['deep', 'learning', 'models', 'used', 'ai', 'systems']

===== 5. 코퍼스 통계 =====
전체 토큰 수 T: